In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# loading the data from csv file to a pandas Dataframe
raw_mail_data = pd.read_csv('Emails.csv')

In [3]:
raw_mail_data

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
...,...,...
5723,Subject: re : research and development charges...,0
5724,"Subject: re : receipts from visit jim , than...",0
5725,Subject: re : enron case study update wow ! a...,0
5726,"Subject: re : interest david , please , call...",0


In [4]:
# replace the null values with a null string
mail_data = raw_mail_data.where((pd.notnull(raw_mail_data)),'')

In [5]:
# printing the first 5 rows of the dataframe
mail_data.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [6]:
# checking the number of rows and columns in the dataframe
mail_data.shape

(5728, 2)

In [7]:
mail_data.isnull().sum().sum()

0

In [8]:
mail_data.duplicated().sum().sum()

33

In [9]:
mail_data=mail_data.drop_duplicates()

In [10]:
mail_data.duplicated().sum().sum()

0

# comarison of all algorithms

In [12]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from joblib import dump, load
from sklearn.preprocessing import LabelEncoder


X = mail_data['text']
Y = mail_data['spam']

# Split the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

# Transform the text data to feature vectors
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)

X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)

# Save the TfidfVectorizer
vectorizer_filename = 'tfidf_vectorizer.joblib'
dump(feature_extraction, vectorizer_filename)
print(f"TfidfVectorizer saved as {vectorizer_filename}")

# Encode labels
label_encoder = LabelEncoder()
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_test_encoded = label_encoder.transform(Y_test)

# Initialize a dictionary to keep track of models and their accuracy
model_accuracies = {}

# SVM Models with Different Kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
for kernel in kernels:
    model = SVC(kernel=kernel)
    model.fit(X_train_features, Y_train_encoded)
    Y_pred = model.predict(X_test_features)
    accuracy = accuracy_score(Y_test_encoded, Y_pred)
    model_accuracies[f"SVM_{kernel}"] = (model, accuracy)
    print(f"SVM with {kernel} kernel accuracy: {accuracy:.4f}")

# Naive Bayes Models
nb_models = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB()
}

for nb_name, nb_model in nb_models.items():
    model = nb_model
    model.fit(X_train_features.toarray(), Y_train_encoded)
    Y_pred = model.predict(X_test_features.toarray())
    accuracy = accuracy_score(Y_test_encoded, Y_pred)
    model_accuracies[nb_name] = (model, accuracy)
    print(f"{nb_name} accuracy: {accuracy:.4f}")

# Logistic Regression
logistic_model = LogisticRegression()
logistic_model.fit(X_train_features, Y_train_encoded)
Y_pred = logistic_model.predict(X_test_features)
accuracy = accuracy_score(Y_test_encoded, Y_pred)
model_accuracies["LogisticRegression"] = (logistic_model, accuracy)
print(f"Logistic Regression accuracy: {accuracy:.4f}")

# Decision Tree Classifier
decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train_features, Y_train_encoded)
Y_pred = decision_tree_model.predict(X_test_features)
accuracy = accuracy_score(Y_test_encoded, Y_pred)
model_accuracies["DecisionTree"] = (decision_tree_model, accuracy)
print(f"Decision Tree accuracy: {accuracy:.4f}")

# Random Forest Classifier
random_forest_model = RandomForestClassifier(n_estimators=100)
random_forest_model.fit(X_train_features, Y_train_encoded)
Y_pred = random_forest_model.predict(X_test_features)
accuracy = accuracy_score(Y_test_encoded, Y_pred)
model_accuracies["RandomForest"] = (random_forest_model, accuracy)
print(f"Random Forest accuracy: {accuracy:.4f}")

# Find the best model based on accuracy
best_model_name = max(model_accuracies, key=lambda k: model_accuracies[k][1])
best_model, best_accuracy = model_accuracies[best_model_name]

# Save the best model
best_model_filename = f"{best_model_name}_best_model.joblib"
dump(best_model, best_model_filename)
print(f"\nBest model is {best_model_name} with accuracy: {best_accuracy:.4f}")
print(f"Model saved as {best_model_filename}")

# Function to load a saved model and vectorizer and predict a new input
def predict_spam(input_text, model_filename, vectorizer_filename):
    # Load the model
    loaded_model = load(model_filename)
    
    # Load the TfidfVectorizer
    loaded_vectorizer = load(vectorizer_filename)
    
    # Transform the input text to feature vector
    input_features = loaded_vectorizer.transform([input_text])
    
    # Predict using the loaded model
    prediction = loaded_model.predict(input_features)[0]
    
    # Output the prediction
    if prediction == 1:
        print(f"The input '{input_text}' is classified as: Spam Mail")
    else:
        print(f"The input '{input_text}' is classified as: Ham Mail")

# Test with a new input
new_input = "Exclusive offer just for you! Click here to claim your prize!"
print("\nTesting with a new input:")
predict_spam(new_input, best_model_filename, vectorizer_filename)


TfidfVectorizer saved as tfidf_vectorizer.joblib
SVM with linear kernel accuracy: 0.9939
SVM with poly kernel accuracy: 0.8464
SVM with rbf kernel accuracy: 0.9895
SVM with sigmoid kernel accuracy: 0.9939
GaussianNB accuracy: 0.9614
MultinomialNB accuracy: 0.9104
BernoulliNB accuracy: 0.9772
Logistic Regression accuracy: 0.9824
Decision Tree accuracy: 0.9508
Random Forest accuracy: 0.9807

Best model is SVM_linear with accuracy: 0.9939
Model saved as SVM_linear_best_model.joblib

Testing with a new input:
The input 'Exclusive offer just for you! Click here to claim your prize!' is classified as: Spam Mail
